In [ ]:
# ───────────────────────────────────────────────────────────────────────
# Célula 1 – imports, logging e path do projeto
# ───────────────────────────────────────────────────────────────────────
import os, sys, importlib, logging, pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s › %(message)s",
    datefmt="%H:%M:%S",
)

PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)


In [ ]:
# ───────────────────────────────────────────────────────────────────────
# Célula 2 – recarregar módulos de código local
# (execute depois de qualquer alteração no projeto)
# ───────────────────────────────────────────────────────────────────────
import extract.sheets_fetcher   as sf_mod
import treat.utils.write_back   as wb_mod
import treat.treat_pipeline     as tp_mod
import treat.treat_runner       as tr_mod
import load.origin_writer       as ow_mod
import load.dest_writer         as dw_mod

for m in (sf_mod, wb_mod, tp_mod, tr_mod, ow_mod, dw_mod):
    importlib.reload(m)


In [ ]:
# ───────────────────────────────────────────────────────────────────────
# Célula 3 – parâmetros globais# ──────────────────────────────────────────────────────────────────────────────
# Célula 2 – recarregar módulos alterados
# ──────────────────────────────────────────────────────────────────────────────
import importlib

import extract.sheets_fetcher   as sf_mod
import treat.utils.write_back   as wb_mod      # agora separado para evitar circular import
import treat.treat_pipeline     as tp_mod
import treat.treat_runner       as tr_mod

importlib.reload(sf_mod)
importlib.reload(wb_mod)
importlib.reload(tp_mod)
importlib.reload(tr_mod)

# ───────────────────────────────────────────────────────────────────────
CREDS_PATH     = "creds.json"
SPREADSHEET_ID = "1DazUQxspLgT0utOFHcTINbFngXw7Fq0LOq6v4lRGixg"

# lista completa de abas de origem
SHEET_NAMES = [
    "metaGeral", "metaIdade", "metaGenero", "metaRegiao", "metaAlcance",
    "tiktokGeral", "tiktokIdade", "tiktokGenero", "tiktokRegiao", "tiktokAlcance",
    "pinterestGeral", "pinterestGenero", "pinterestIdade", "pinterestRegiao", "pinterestAlcance",
    "linkedinGeral", "LinkedinRegiao", "linkedinAlcance",
]

# se quiser gravar de verdade, troque para True
WRITE_BACK = False


In [ ]:
# ───────────────────────────────────────────────────────────────────────
# Célula 4 – helpers
# ───────────────────────────────────────────────────────────────────────
from extract.sheets_fetcher        import SheetsFetcher
from treat.treat_pipeline          import TreatPipeline
from treat.utils.renomeacoes       import renomeacao_geral, renomear_colunas_origem_para_modelo
from treat.utils.campos_calculados import calcular_engajamento_total, gerar_id
from load.origin_writer            import write_back_origin
from load.dest_writer              import write_back_for_sheet

fetcher = SheetsFetcher(SPREADSHEET_ID, CREDS_PATH)

def run_etl_for_sheet(sheet_name: str, *, write_back: bool = False) -> dict[str, pd.DataFrame]:
    """Executa o pipeline completo para uma aba e devolve DataFrames intermediários."""
    # 1) extrai dados crus
    df_raw = fetcher.get([sheet_name])[sheet_name]

    # 2) trata (sem gravar ainda)
    pipeline = TreatPipeline(
        creds_path         = CREDS_PATH,
        spreadsheet_id     = SPREADSHEET_ID,
        sheet_name         = sheet_name,
        mapping_renomeacao = renomeacao_geral,
        write_back         = write_back,
    )
    df_ok = pipeline.run(df_raw)

    # 3) write-back na origem (dry_run respeita flag write_back)
    df_origin = write_back_origin(
        df_raw         = df_raw,
        df_ok          = df_ok,
        creds_path     = CREDS_PATH,
        spreadsheet_id = SPREADSHEET_ID,
        sheet_name     = sheet_name,
        write_back     = write_back,
        dry_run        = not write_back,
    ) or pd.DataFrame()  # pode vir None se nada mudou

    # 4) transforma para modelo
    df_model = renomear_colunas_origem_para_modelo(df_ok, renomeacao_geral)
    df_model = calcular_engajamento_total(df_model)
    df_model["ID"] = df_model.apply(gerar_id, axis=1)

    # 5) write-back destino (dry_run idem)
    df_dest = write_back_for_sheet(
        df_model        = df_model,
        sheet_name      = sheet_name,
        creds_path      = CREDS_PATH,
        spreadsheet_id  = SPREADSHEET_ID,
        write_back      = write_back,
        dry_run         = not write_back,
    ) or pd.DataFrame()

    return {
        "raw":    df_raw,
        "ok":     df_ok,
        "origin": df_origin,
        "model":  df_model,
        "dest":   df_dest,
    }


In [ ]:
# ───────────────────────────────────────────────────────────────────────
# Célula 5 – loop principal
# ───────────────────────────────────────────────────────────────────────
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 5)

results = {}

for sheet in SHEET_NAMES:
    print(f"\n▶️ Processando '{sheet}' …")
    dfs = run_etl_for_sheet(sheet, write_back=WRITE_BACK)
    results[sheet] = dfs

    # Mostra só shapes para visão rápida
    shapes = {k: v.shape for k, v in dfs.items()}
    print("   ", shapes)


In [ ]:
# ───────────────────────────────────────────────────────────────────────
# Célula 6 – inspeção opcional de um DataFrame específico
# ───────────────────────────────────────────────────────────────────────
# Exemplo: mostrar as 3 primeiras linhas do modelo gerado para tiktokGenero
sheet   = "tiktokGenero"
stage   = "model"          # raw | ok | origin | model | dest
display(results[sheet][stage].head(3))


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Célula X – testar write-back na origem em dry run
# ──────────────────────────────────────────────────────────────────────────────

from load.origin_writer import write_back_origin

# executa em dry_run=True para não tocar no Sheets de verdade
df_wb = write_back_origin(
    df_raw          = df_raw,
    df_ok           = df_ok,
    creds_path      = creds_path,
    spreadsheet_id  = spreadsheet_id,
    sheet_name      = sheet_name,
    write_back      = True,    # liga a lógica de write-back
    dry_run         = False,    # mas evita alteração real
)

print(f"▶️ Dry-run: {df_wb.shape[0]} linhas × {df_wb.shape[1]} colunas preparadas")
df_wb.head(5)


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Celula Y – recarregar dest_writer e importar função
# ──────────────────────────────────────────────────────────────────────────────

import importlib
import load.dest_writer as dw_mod
importlib.reload(dw_mod)

from load.dest_writer import write_back_for_sheet


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Célula Z – testar write-back destino em dry-run
# ──────────────────────────────────────────────────────────────────────────────

# 1) Preparar o DataFrame no formato do modelo
from treat.utils.renomeacoes import renomear_colunas_origem_para_modelo
from treat.utils.campos_calculados import calcular_engajamento_total
from treat.utils.campos_calculados import gerar_id

df_model = renomear_colunas_origem_para_modelo(df_ok, renomeacao_geral)
df_model = calcular_engajamento_total(df_model)
df_model["ID"] = df_model.apply(gerar_id, axis=1)

# 2) Dry-run do write-back na aba de destino
from load.dest_writer import write_back_for_sheet

df_dest = write_back_for_sheet(
    df_model        = df_model,
    sheet_name      = sheet_name,    # ex.: "tiktokAlcance"
    creds_path      = creds_path,
    spreadsheet_id  = spreadsheet_id,
    write_back      = True,          # habilita a rotina
    dry_run         = True,          # evita chamada real ao Sheets
)

print(f"▶️ Dry-run destino: {df_dest.shape[0]} linhas × {df_dest.shape[1]} colunas")
df_dest.head(5)


In [ ]:
import importlib
import treat.bi_param_utils as bp
importlib.reload(bp)

import importlib, utils.preview_links as pl
importlib.reload(pl)


df_ok = pipeline.run(df_raw)
import importlib, treat.treat_pipeline as tp
importlib.reload(tp)



In [ ]:
df_written = write_back_for_sheet(
    df_model        = df_model,
    sheet_name      = sheet_name,
    creds_path      = creds_path,
    spreadsheet_id  = spreadsheet_id,
    write_back      = True,
    dry_run         = False,   # grava de verdade!
)

if df_written is not None:
    print(f"✅ {len(df_written)} linhas gravadas na aba {sheet_name}.")
else:
    print("⚠️ Nenhuma linha nova para gravar.")
